# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal1 = con.sql(f"""
SELECT
    CASE WHEN avg_position <= 10 THEN 'top_10' ELSE 'beyond_10' END as position_bucket,
    COUNT(*) as n,
    AVG(ctr) as avg_ctr
FROM (
    SELECT
        gsc_clicks * 1.0 / gsc_impressions as ctr,
        gsc_sum_position * 1.0 / gsc_impressions as avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
)
GROUP BY position_bucket
""").df()

signal1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,top_10,2183484,0.003900
1,beyond_10,1427577,0.001828


Signal 1 — CTR vs. position (behind CTR-fix logic): Verdict: CONFIRMED.
top_10 (n=2,183,484) shows an average CTR of 0.0039, while beyond_10 (n=1,427,577) shows 0.0018 — roughly 2.1x higher CTR for top-10 positions. This confirms the expected direction behind FlyRank's CTR-fix logic: higher-ranked pages earn meaningfully more clicks per impression.

Note: these real CTR values (under 1%) are far smaller than the values I used in my earlier ML-02/ML-03 work with the small starter CSV (which showed ~78%/26%). This is an important correction — the anonymized starter sample was not representative of real CTR scale, and my baseline rule below uses these corrected, real numbers instead.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal2 = con.sql(f"""
WITH base AS (
    SELECT
        gsc_impressions,
        gsc_clicks * 1.0 / gsc_impressions as ctr,
        gsc_sum_position * 1.0 / gsc_impressions as avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
),
med AS (SELECT MEDIAN(gsc_impressions) as m FROM base)
SELECT
    CASE WHEN base.gsc_impressions >= med.m THEN 'high_volume' ELSE 'low_volume' END as volume_bucket,
    COUNT(*) as n,
    AVG(base.ctr) as avg_ctr,
    AVG(base.avg_position) as avg_position
FROM base, med
GROUP BY volume_bucket
""").df()

signal2


,volume_bucket,n,avg_ctr,avg_position
0,low_volume,1799187,0.003276,19.819652
1,high_volume,1811874,0.002887,11.861610


Signal 2 — Volume (behind quick-win logic): Verdict: MIXED.
high_volume pages (n=1,811,874) have a notably better average position (11.86) than low_volume pages (n=1,799,187, avg_position=19.82) — volume correlates with ranking strength, as expected. However, high_volume pages show a slightly lower average CTR (0.0029) than low_volume pages (0.0033), despite their better position. This is counterintuitive: if position alone drove CTR (per Signal 1), better-positioned high-volume pages should show higher CTR, not lower.

This MIXED result suggests volume is a decent proxy for position, but not a reliable standalone signal for CTR performance — a high-volume page in a strong position may still underperform its expected CTR, which is exactly the kind of gap my baseline rule (Section 2) is designed to catch. Volume alone should not be used as a quick-win filter without accounting for the CTR-vs-position relationship from Signal 1.

In [10]:
import pandas as pd
import os

rule_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks * 1.0 / gsc_impressions as ctr,
    gsc_sum_position * 1.0 / gsc_impressions as avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

rule_df['expected_ctr'] = rule_df['avg_position'].apply(lambda p: 0.0039 if p <= 10 else 0.0018)
rule_df['score'] = rule_df['expected_ctr'] - rule_df['ctr']
rule_df['reason_code'] = 'CTR_BELOW_EXPECTED_FOR_POSITION'
rule_df['action'] = rule_df.apply(
    lambda r: 'REVIEW_TITLE_META' if r['score'] > 0.001 else 'MONITOR', axis=1
)

ranked_queue = rule_df.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

ranked_queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,ctr,avg_position,expected_ctr,score,reason_code,action
0,client_20259bd6705d81d4,content_e59204eb82d02e13,23,0.0,0.956522,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
1,client_20259bd6705d81d4,content_d03c5b437aa172e5,10,0.0,4.900000,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
2,client_20259bd6705d81d4,content_c5d5282bba9a3f18,6,0.0,1.166667,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
3,client_20259bd6705d81d4,content_f29cdaa1740a8e11,49,0.0,7.612245,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
4,client_20259bd6705d81d4,content_50f20b2fa9607f2f,377,0.0,4.689655,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
5,client_20259bd6705d81d4,content_aff3c07fb1ddf8d6,43,0.0,0.790698,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
6,client_20259bd6705d81d4,content_8f0ec140fa9ce9a5,6,0.0,5.000000,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
7,client_73cda7b4e4f265ea,content_91ffe8aef1f8c426,7,0.0,2.285714,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
8,client_73cda7b4e4f265ea,content_03940665a88bf663,1,0.0,0.000000,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
9,client_73cda7b4e4f265ea,content_30fc0ffeed8d67e6,73,0.0,7.780822,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Top-10 review (skeptic's read):

All ten rows share the same score (0.0039) and reason_code (CTR_BELOW_EXPECTED_FOR_POSITION) — every one has ctr=0.0, the maximum possible gap for a top-10-position page. This immediately reveals a rule weakness: when many pages tie at the same maximum score, the rule provides no way to rank within the tie, and low-impression rows dominate the top purely because 0 clicks out of very few impressions is common and easy to hit.

Row 1 (23 impressions, avg_position 0.96): Action REVIEW_TITLE_META. Why: 0 clicks despite ranking #1 on average. What would make it wrong: 23 impressions is a small sample — a single missed click or two could flip this from 0% to a normal CTR; this may just be noise, not a real problem page.

Row 2 (10 impressions, avg_position 4.9): Same action/reason. What would make it wrong: only 10 impressions — far too little volume to trust as a genuine signal.

Row 3 (6 impressions): What would make it wrong: extremely low volume (6 impressions) makes this essentially untrustworthy as a ranked "top priority" — a minimum impression threshold is clearly needed.

Rows 4-6 (49, 377, 43 impressions): These have more reasonable volume, and 377 impressions with 0 clicks is a genuinely strong signal worth a human review — this is the kind of row the baseline should be surfacing.

Rows 7-10 (7, 1, 73 impressions, plus a repeat of client_73cda7b4e4f265ea): Similar low-volume issue; row 9 has just 1 impression — this should not appear in a "top priority" queue at all.

Overall skeptic's take: the rule correctly identifies the direction (0 CTR is bad), but it has no volume floor, so it's currently dominated by statistically unreliable, low-impression rows rather than the genuinely important cases like row 5 (377 impressions, 0 clicks).


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



Fields I excluded and why:

1. gsc_sum_position (raw) — excluded as a standalone feature; I only use it derived (avg_position = sum_position/impressions), since the raw sum has no meaning on its own without dividing by impressions.

2. client_has_gsc / client_has_ga4 — excluded from the rule itself (only used earlier for filtering gsc_data_available). These are access flags, not performance signals, so including them directly in the score would conflate "do we have data" with "is this page underperforming."

3. AI referral columns (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other) — excluded because they're a separate traffic channel from organic search CTR, and mixing them into a search-position-based rule would blur what the rule is actually measuring.

4. sessions_ai, scroll_events — excluded as engagement metrics that happen after a click, not before it; my rule is about earning the click in the first place (CTR), not what happens once someone lands on the page.

5. client_hash_id / content_hash_id — excluded from the scoring logic itself (used only as identifiers to track which row is which), since including a hashed ID as a model input risks the model learning client-specific quirks rather than a generalizable pattern.

6. ga4_data_available and any GA4-derived metrics — excluded because not all clients have GA4 access (per dim_clients), so relying on it would bias the rule toward only GA4-enabled clients, contradicting the "unbalanced history" limitation I noted in ML-04.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.